##Predicting Product Recommendations from Women’s E-Commerce Clothing Reviews

This project uses a women’s e-commerce clothing reviews dataset to predict whether a customer will recommend a product. The dataset includes numeric, categorical, and text features, which makes it a strong fit for this assignment. A machine learning classification model is built using structured review information and review text.

###Project Overview

The goal of this project is to predict the variable Recommended IND, which indicates whether a customer recommended a clothing product.

This dataset contains:

* Numeric features: Age, Positive Feedback Count, Review_Length
*  Categorical features: Division Name, Department Name, Class Name
*  Text feature: Review Text
*  Label: Recommended IND


This project satisfies the assignment requirement to include numeric variables, categorical variables, one text-based feature, and a prediction label.

##Import Libraries

This section imports the Python libraries needed for data loading, preprocessing, modeling, and evaluation.

In [16]:
from google.colab import files
uploaded = files.upload()

Saving Womens Clothing E-Commerce Reviews.csv to Womens Clothing E-Commerce Reviews (1).csv


##Load the Dataset

The dataset is loaded into a pandas DataFrame so it can be inspected and prepared for analysis.

In [17]:
import pandas as pd

df = pd.read_csv('Womens Clothing E-Commerce Reviews.csv')
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


##Inspect the Data

Before modeling, it is important to understand the structure of the dataset, including the available columns and the first few rows.

In [18]:
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

##Select Relevant Variables

Only the variables needed for the project are retained. These columns support the numeric, categorical, text, and label requirements of the assignment.

In [19]:
df = df[['Age', 'Review Text', 'Recommended IND', 'Positive Feedback Count',
         'Division Name', 'Department Name', 'Class Name']]

df.head()

,Age,Review Text,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,33,Absolutely wonderful - silky and sexy and comf...,1,0,Initmates,Intimate,Intimates
1,34,Love this dress! it's sooo pretty. i happene...,1,4,General,Dresses,Dresses
2,60,I had such high hopes for this dress and reall...,0,0,General,Dresses,Dresses
3,50,"I love, love, love this jumpsuit. it's fun, fl...",1,0,General Petite,Bottoms,Pants
4,47,This shirt is very flattering to all due to th...,1,6,General,Tops,Blouses


##Clean Missing Values

Missing values in the text and categorical fields are removed to ensure the model is trained on complete records.

In [20]:
df = df.dropna(subset=['Review Text', 'Division Name', 'Department Name', 'Class Name'])
df = df.reset_index(drop=True)
df.shape

(22628, 7)

##Feature Engineering

A new numeric feature called Review_Length is created by calculating the number of characters in each review. This gives the model an additional measure of how detailed a customer’s review is.

In [21]:
df['Review_Length'] = df['Review Text'].apply(len)
df.head()

,Age,Review Text,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name,Review_Length
0,33,Absolutely wonderful - silky and sexy and comf...,1,0,Initmates,Intimate,Intimates,53
1,34,Love this dress! it's sooo pretty. i happene...,1,4,General,Dresses,Dresses,303
2,60,I had such high hopes for this dress and reall...,0,0,General,Dresses,Dresses,500
3,50,"I love, love, love this jumpsuit. it's fun, fl...",1,0,General Petite,Bottoms,Pants,124
4,47,This shirt is very flattering to all due to th...,1,6,General,Tops,Blouses,192


##Define Features and Label

The predictor variables are stored in X, and the target variable Recommended IND is stored in y.

In [22]:
X = df[['Age', 'Positive Feedback Count', 'Review_Length',
        'Division Name', 'Department Name', 'Class Name', 'Review Text']]

y = df['Recommended IND']

##Split the Data


The dataset is divided into training and testing sets. The training set is used to train the model, while the testing set is used to evaluate prediction performance on unseen data.

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

##Preprocess the Data

Different types of data require different preprocessing methods.


*   Numeric variables are imputed and scaled
*   Categorical variables are imputed and one-hot encoded
*   Text data is transformed using TF-IDF vectorization


A ColumnTransformer is used to apply all preprocessing steps in one pipeline.

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

numeric_features = ['Age', 'Positive Feedback Count', 'Review_Length']
categorical_features = ['Division Name', 'Department Name', 'Class Name']
text_feature = 'Review Text'

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('txt', TfidfVectorizer(stop_words='english', max_features=3000), text_feature)
    ]
)

##Build the Model

A logistic regression classifier is used for this project. It is a strong baseline model for binary classification problems and works well with TF-IDF text features.

In [25]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

##Train the Model

The model is trained using the training portion of the dataset.

In [26]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age',
                                                   'Positive Feedback Count',
                                                   'Review_Length']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Division Name',
                                                   'Department Name',
                                                   'Class Name']),
                                                 ('txt',
                                                  TfidfVectorizer(max_features=3000,
                                                                  stop_words='english'),
                                                  'Review Text')])),
                ('classifier', LogisticRegression(max_iter=1000))])

##Evaluate the Model

Model performance is evaluated using accuracy, a classification report, and a confusion matrix. These measures show how well the model predicts whether a product is recommended.

In [27]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8864339372514362

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.52      0.62       820
           1       0.90      0.97      0.93      3706

    accuracy                           0.89      4526
   macro avg       0.84      0.74      0.78      4526
weighted avg       0.88      0.89      0.88      4526


Confusion Matrix:
 [[ 427  393]
 [ 121 3585]]


The model produced an accuracy score of 0.8864.

The classification report shows that the model predicts the recommended class very well, while performance for the non-recommended class is lower due to class imbalance.

The confusion matrix shows that most predictions were correct, with only a small number of misclassified reviews.

In [28]:
df.to_csv("cleaned_clothing_reviews.csv", index=False)

In [29]:
from google.colab import files
files.download("cleaned_clothing_reviews.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Conclusion

This project demonstrated how numeric, categorical, and text data can be combined in one machine learning workflow.

Using customer review information and review text, the model successfully predicted whether a customer would recommend a clothing product.

This project satisfies the assignment requirements by including numeric features, categorical features, text features, and a prediction label.